# La prosa enciclopedica regge il protocollo di Altmann?

**Test di fattibilità, da eseguire prima di qualunque confronto Wikipedia / Grokipedia.**

L'idea di confrontare un'enciclopedia scritta da un LLM con Wikipedia è attraente: risolve il
problema della lunghezza, non ha degenerazione, è appaiabile per titolo e costa zero. Ma poggia
su un presupposto che va verificato per primo:

> l'effetto di Altmann — le keyword più bursty dei controlli appaiati in frequenza — **esiste**
> nella prosa enciclopedica?

Il meccanismo del paper è la persistenza tematica *narrativa*: il romanzo indugia su un
personaggio per un capitolo e poi passa ad altro, e questo genera code larghe in $p(\tau)$. Una
voce enciclopedica su Napoleone nomina Napoleone in modo tendenzialmente **uniforme** dall'inizio
alla fine: è stazionaria, cioè l'opposto di bursty. Se l'effetto non c'è nemmeno in Wikipedia,
non c'è nessun segnale che il confronto con Grokipedia possa rilevare, e l'intera strada è morta
prima di partire.

Questo notebook misura quel presupposto su alcune decine di articoli lunghi, con **lo stesso
codice validato** del notebook principale, e lo confronta con i segmenti di *Guerra e pace* alla
stessa lunghezza.

---

## Come sono scelti gli articoli

Non per byte di wikitext. Ordinare Wikipedia per dimensione restituisce liste, tabelle
elettorali e stagioni televisive: una pagina da 679.016 byte può contenere **1.284 caratteri**
di prosa. Nemmeno i Featured Article vanno bene: sono scelti per qualità, non per lunghezza, e
un campione casuale restituisce stagioni calcistiche ed episodi di serie TV.

Gli articoli a tema ampio invece hanno la lunghezza e la densità di frasi giuste (mediana
~70.000 caratteri di prosa, 6–8 frasi ogni 1000 caratteri, contro le 8,6 del testo letterario).
Il pool di partenza è quindi una lista esplicita di argomenti ampi, che viene poi **filtrata con
gli stessi gate di qualità** del notebook principale e ordinata per lunghezza di prosa reale.
La lista è in cima alla cella di configurazione ed è modificabile.

## Cosa aspettarsi

Il risultato non è scontato in nessuna delle due direzioni. Il paper nuovo di Degli Esposti e
Montemurro include nel suo corpus il *Principia* di Newton e *The Analysis of Mind* di Russell,
cioè prosa espositiva, e per entrambi trova $\alpha > 0.7$: l'espositivo funziona. Ma sono opere
di libro, di autore singolo e con un filo argomentativo continuo, non voci enciclopediche
scritte da molte mani e organizzate in sezioni indipendenti.

## 1. Configurazione

In [ ]:
from pathlib import Path

# --- dove sta la cartella dei dati (il livello sopra questo notebook) ---
QUI = Path.cwd()
BASE = QUI if (QUI / "corpora_cache").is_dir() else QUI.parent
CACHE_WIKI = QUI / "cache_wikipedia"      # estratti scaricati
CACHE_LIB  = BASE / "corpora_cache"       # cache condivisa col notebook principale
OUT_DIR    = QUI / "risultati_wiki"

# --- corpus letterario di riferimento (identico al notebook principale) ---
GUTENBERG_URL = "https://www.gutenberg.org/cache/epub/2600/pg2600.txt"
START_PHRASE  = "Well, Prince, so Genoa and Lucca"
PROMPT_CHARS  = 8262

# --- lunghezze: IDENTICHE al notebook principale, per poter confrontare i numeri ---
N_EFF           = 50_000   # troncamento comune a tutti i testi
N_ARTICOLI      = 40       # quanti articoli tenere (i piu' lunghi che passano i gate)
N_SEG_LETTERARI = 20       # segmenti di Guerra e pace
MIN_EVENTS      = 15

LAG_MIN, LAG_MAX  = 4, 4000
N_LAGS            = 40
FIT_RANGE         = (100, 2000)
FIT_RANGE_STRETTO = (100, 500)

# --- selezione dei bersagli (wordset "auto": riselezionati dentro ogni testo) ---
N_LETTERS, N_FUNCTION, N_KEYWORDS, N_MATCHED = 19, 6, 7, 7
PROPER_CAP_RATIO = 0.6
N_NULL_REPS = 3

# --- gate di qualita' della prosa (calibrati sul riferimento letterario) ---
FRASI_1K_MIN   = 3.0    # frasi per 1000 caratteri; letterario ~8.6
RIGHE_CORTE_MAX = 0.45  # frazione di righe < 80 caratteri: alta = liste/tabelle
LEN_TOKEN_MAX  = 6.0    # lunghezza media dei token; letterario ~4.5

RANDOM_SEED = 20260813
DPI = 150

# --- pool di partenza: argomenti ampi, quelli che hanno prosa lunga davvero ---
ARGOMENTI = """
Napoleon; World War II; World War I; United States; India; China; Japan; Russia; Germany;
France; United Kingdom; Italy; Brazil; Canada; Australia; Roman Empire; Byzantine Empire;
Ottoman Empire; Mongol Empire; British Empire; Ancient Egypt; Ancient Greece; Ancient Rome;
Middle Ages; Renaissance; French Revolution; American Civil War; Cold War; Industrial Revolution;
Abraham Lincoln; Albert Einstein; Isaac Newton; Charles Darwin; Winston Churchill; Adolf Hitler;
Joseph Stalin; Mahatma Gandhi; Nelson Mandela; Genghis Khan; Julius Caesar; Alexander the Great;
Leonardo da Vinci; William Shakespeare; Wolfgang Amadeus Mozart; Ludwig van Beethoven;
Pablo Picasso; Vincent van Gogh; Elizabeth II; Barack Obama; Karl Marx; Sigmund Freud;
Christianity; Islam; Buddhism; Hinduism; Judaism; Catholic Church; Protestantism;
Philosophy; Science; Mathematics; Physics; Chemistry; Biology; Astronomy; Geology; Medicine;
Evolution; Genetics; Quantum mechanics; Theory of relativity; Climate change; Universe;
Earth; Moon; Sun; Solar System; Human; Brain; Language; Music; Literature; Art; Architecture;
History; Economics; Capitalism; Socialism; Democracy; Law; War; Slavery; Colonialism;
New York City; London; Paris; Rome; Beijing; Moscow; Tokyo; Delhi; Istanbul; Cairo;
Internet; Computer; Artificial intelligence; Television; Film; Photography; Printing press;
Agriculture; Industrial Revolution; Railway; Automobile; Aviation; Space exploration;
Apollo program; Nuclear weapon; Antibiotic; Vaccine; DNA; Cancer; Pandemic;
Africa; Europe; Asia; South America; Antarctica; Ocean; Amazon rainforest; Himalayas;
Football; Olympic Games; Chess; Wikipedia; Steve Jobs; Henry Ford; Scientific method
"""
ARGOMENTI = [t.strip() for t in ARGOMENTI.replace("\n", " ").split(";") if t.strip()]
print(f"argomenti nel pool: {len(ARGOMENTI)}")
print("cartella dati:", BASE.resolve())

In [ ]:
import json, re, ssl, math, time, hashlib, random, warnings
import urllib.request, urllib.parse
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

try:
    from scipy import stats as sps
    HAS_SCIPY = True
except Exception:
    HAS_SCIPY = False
try:
    import certifi
    HAS_CERTIFI = True
except Exception:
    HAS_CERTIFI = False

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": DPI, "savefig.bbox": "tight",
                     "font.size": 10, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.axisbelow": True, "legend.frameon": True})
warnings.filterwarnings("ignore", category=RuntimeWarning)

for d in (CACHE_WIKI, OUT_DIR, OUT_DIR / "figure", OUT_DIR / "dati"):
    d.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(RANDOM_SEED)
LAGS = np.unique(np.logspace(np.log10(LAG_MIN), np.log10(LAG_MAX), N_LAGS).astype(int))
print("lag:", LAGS[:5], "...", LAGS[-2:], f"({len(LAGS)})")
print("scipy:", HAS_SCIPY, "| certifi:", HAS_CERTIFI, "| output ->", OUT_DIR.resolve())

## 2. Il nucleo di misura

Copiato **verbatim** dal notebook principale, dove è già validato contro il paper
($\sigma_\tau/\langle\tau\rangle = 0.844$ per la lettera "e" contro 0.83 di Altmann,
3.892 per *prince* contro 3.86, $\hat\gamma_{A1} = 0.98$). Non va modificato qui: se i
due notebook divergono, i numeri non sono più confrontabili.

In [ ]:
WORD_CHAR = r"[^\W\d_]"
TOKEN_RE  = re.compile(WORD_CHAR + r"+(?:['\u2019\-]" + WORD_CHAR + r"+)*", re.UNICODE)
SENT_END  = set(".!?")

def build_char_array(text):
    return np.array(list(text.lower()))

def pos_from_char(carr, ch):
    return np.flatnonzero(carr == ch)

def pos_from_set(carr, chars):
    return np.flatnonzero(np.isin(carr, list(chars)))

_word_re_cache = {}
def pos_from_word(text, w):
    if w not in _word_re_cache:
        _word_re_cache[w] = re.compile(
            r"(?<!" + WORD_CHAR + r")" + re.escape(w) + r"(?!" + WORD_CHAR + r")",
            re.IGNORECASE | re.UNICODE)
    return np.fromiter((m.start() for m in _word_re_cache[w].finditer(text)), dtype=np.int64)

def seme(*parti):
    h = hashlib.sha256("|".join(map(str, parti)).encode("utf-8")).hexdigest()
    return (int(h[:8], 16) ^ RANDOM_SEED) % (2 ** 32)

def transport_sigma2(pos, N, lags=None):
    lags = LAGS if lags is None else lags
    x = np.zeros(N, dtype=np.float64)
    if len(pos):
        x[pos[pos < N]] = 1.0
    C = np.concatenate(([0.0], np.cumsum(x)))
    out = np.full(len(lags), np.nan)
    for i, t in enumerate(lags):
        if t >= N // 4:
            continue
        d = C[t:] - C[:-t]
        out[i] = d.var()
    return out

def fit_gamma(lags, s2, lo, hi, min_pts=5):
    lags = np.asarray(lags, float); s2 = np.asarray(s2, float)
    m = np.isfinite(s2) & (s2 > 0) & (lags >= lo) & (lags <= hi)
    if m.sum() < min_pts:
        return np.nan, np.nan
    X, Y = np.log10(lags[m]), np.log10(s2[m])
    b, a = np.polyfit(X, Y, 1)
    resid = Y - (b * X + a)
    sxx = ((X - X.mean()) ** 2).sum()
    se = float(np.sqrt((resid ** 2).sum() / max(len(X) - 2, 1) / sxx)) if sxx > 0 else np.nan
    return float(b), se

def interevent(pos):
    return np.diff(np.asarray(pos, dtype=np.int64)) if len(pos) > 1 else np.array([], np.int64)

def burstiness_stats(tau):
    if len(tau) < 2:
        return dict(mean_tau=np.nan, sigma_tau=np.nan, cv_tau=np.nan, B_goh=np.nan)
    m, s = float(tau.mean()), float(tau.std(ddof=1))
    return dict(mean_tau=m, sigma_tau=s, cv_tau=s / m if m > 0 else np.nan,
                B_goh=(s - m) / (s + m) if (s + m) > 0 else np.nan)

def null_A1(pos, N, r):
    M = len(pos)
    return np.sort(r.choice(N, size=min(M, N), replace=False)) if M else pos

def null_A2(pos, N, r):
    tau = interevent(pos)
    if len(tau) < 2:
        return pos
    tau = r.permutation(tau)
    new = np.concatenate(([pos[0]], pos[0] + np.cumsum(tau)))
    return new[new < N]

def analyze_sequence(pos, N, label, level, r, n_null=N_NULL_REPS,
                     fit_range=FIT_RANGE):
    pos = np.asarray(pos, dtype=np.int64); pos = pos[pos < N]
    M = len(pos)
    rec = dict(sequenza=label, livello=level, n_eventi=M,
               frequenza=M / N if N else np.nan, n_chars=N)
    if M < MIN_EVENTS:
        rec.update(dict(gamma=np.nan, gamma_se=np.nan, gamma_stretto=np.nan,
                        gamma_A1=np.nan, gamma_A2=np.nan, mean_tau=np.nan,
                        sigma_tau=np.nan, cv_tau=np.nan, B_goh=np.nan, stimabile=False))
        return rec
    s2 = transport_sigma2(pos, N)
    g, se = fit_gamma(LAGS, s2, *fit_range)
    gs, _ = fit_gamma(LAGS, s2, *FIT_RANGE_STRETTO)
    rec.update(dict(gamma=g, gamma_se=se, gamma_stretto=gs, stimabile=True))
    rec.update(burstiness_stats(interevent(pos)))
    g1, g2 = [], []
    for _ in range(n_null):
        g1.append(fit_gamma(LAGS, transport_sigma2(null_A1(pos, N, r), N), *fit_range)[0])
        g2.append(fit_gamma(LAGS, transport_sigma2(null_A2(pos, N, r), N), *fit_range)[0])
    rec["gamma_A1"] = float(np.nanmean(g1)); rec["gamma_A2"] = float(np.nanmean(g2))
    return rec

STOPWORDS = set("""a about above after again against all am an and any are as at be because been
before being below between both but by can cannot could did do does doing down during each few for
from further had has have having he her here hers herself him himself his how i if in into is it its
itself me more most my myself no nor not of off on once only or other ought our ours ourselves out
over own same she should so some such than that the their theirs them themselves then there these
they this those through to too under until up very was we were what when where which while who whom
why with would you your yours yourself yourselves said one two would shall may might must upon""".split())
VOWELS = set("aeiou")

def word_stats(text):
    freq, cap, tot = Counter(), Counter(), Counter()
    prev_end, primo = 0, True
    for m in TOKEN_RE.finditer(text):
        w = m.group(0); lw = w.lower()
        freq[lw] += 1
        gap = text[prev_end:m.start()]
        inizio = primo or any(c in SENT_END for c in gap) or "\n\n" in gap
        if not inizio:
            tot[lw] += 1
            if w[0].isupper():
                cap[lw] += 1
        prev_end, primo = m.end(), False
    return freq, {w: (cap[w] / tot[w] if tot[w] >= 3 else 0.0) for w in freq}

def select_targets(text):
    low = text.lower()
    letters = [c for c, _ in Counter(c for c in low if c.isalpha() and c.isascii())
               .most_common(N_LETTERS)]
    freq, capr = word_stats(text)
    ordered = [w for w, _ in freq.most_common() if len(w) >= 2]
    funcs = [w for w in ordered if w in STOPWORDS][:N_FUNCTION]
    def is_proper(w): return capr.get(w, 0) >= PROPER_CAP_RATIO
    def is_key(w):    return is_proper(w) or (w not in STOPWORDS and len(w) >= 4)
    keys = [w for w in ordered if is_key(w)][:N_KEYWORDS]
    kset = set(keys)
    pool = [w for w in ordered if w not in kset and not is_proper(w) and w not in funcs]
    matched, used = [], set()
    for k in keys[:N_MATCHED]:
        fk = freq[k]
        cand = sorted((w for w in pool if w not in used),
                      key=lambda w: (abs(math.log((freq[w] + 1e-9) / (fk + 1e-9))), w))
        if cand:
            matched.append(cand[0]); used.add(cand[0])
    return {"vc": [("vocali", "vocali", VOWELS)],
            "lettere": [("spazio", "spazio", " ")] + [(c, "lettera", c) for c in letters],
            "parole": ([(w, "funzione", w) for w in funcs] +
                       [(w, "keyword", w) for w in keys] +
                       [(w, "appaiata", w) for w in matched])}, freq

def targets_positions(text, targets):
    carr = build_char_array(text)
    out = []
    for lev, items in targets.items():
        for lab, tp, key in items:
            if lev == "vc":       pos = pos_from_set(carr, key)
            elif tp == "spazio":  pos = pos_from_char(carr, " ")
            elif tp == "lettera": pos = pos_from_char(carr, key)
            else:                 pos = pos_from_word(text, key)
            out.append((lab, tp, lev, pos))
    return out

def analizza_testo(text, meta):
    """wordset auto: i bersagli sono riselezionati dentro questo testo"""
    N = min(len(text), N_EFF); t = text[:N]
    r = np.random.default_rng(seme(meta["label"], "auto"))
    tg, _ = select_targets(t)
    recs = []
    for lab, tp, lev, pos in targets_positions(t, tg):
        rec = analyze_sequence(pos, N, lab, lev, r)
        rec.update(tipo=tp, **meta)
        recs.append(rec)
    return recs

print("nucleo di misura caricato")

## 3. Scaricamento degli articoli

Tre dettagli che sono bug se sbagliati:

- **SSL** — su questa installazione `ssl.create_default_context()` fallisce con
  `[ASN1: NOT_ENOUGH_DATA]` perché il caricamento dei certificati dallo store di Windows è
  rotto: si passa il bundle PEM di `certifi`.
- **Un titolo per richiesta** — l'API restituisce un solo estratto completo per richiesta anche
  se se ne chiedono venti: i titoli successivi tornano vuoti in silenzio. Chi fa batching
  ottiene un dataset in cui il 95% degli articoli è una stringa vuota.
- **Cache in binario** — `write_text` su Windows traduce i newline e raddoppia gli a-capo alla
  rilettura. Qui il tempo si conta in caratteri, quindi si legge e si scrive in byte.

In [ ]:
API = "https://en.wikipedia.org/w/api.php"
UA  = "AltmannBurstinessResearch/1.0 (tesi universitaria) python-urllib"

def contesto_ssl():
    if HAS_CERTIFI:
        try:
            return ssl.create_default_context(cafile=certifi.where())
        except Exception:
            pass
    try:
        return ssl.create_default_context()
    except ssl.SSLError:
        ctx = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
        ctx.check_hostname = False; ctx.verify_mode = ssl.CERT_NONE
        print("ATTENZIONE: verifica del certificato disattivata; installare certifi")
        return ctx

CTX = contesto_ssl()

def api(pausa=0.3, tentativi=6, **params):
    params.setdefault("format", "json"); params.setdefault("formatversion", "2")
    url = API + "?" + urllib.parse.urlencode(params)
    for k in range(tentativi):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": UA})
            with urllib.request.urlopen(req, timeout=60, context=CTX) as r:
                d = json.loads(r.read().decode("utf-8"))
            time.sleep(pausa)
            return d
        except urllib.error.HTTPError as e:
            if e.code == 429:              # rate limit: attesa esponenziale
                time.sleep(1.5 * (2 ** k)); continue
            raise
    raise RuntimeError("troppi errori 429: rallentare `pausa`")

def scarica(titolo):
    """estratto in testo semplice, con le intestazioni marcate come == Titolo ==."""
    fn = CACHE_WIKI / (hashlib.sha256(titolo.encode()).hexdigest()[:14] + ".txt")
    if fn.exists():
        return fn.read_bytes().decode("utf-8", errors="replace")
    d = api(action="query", prop="extracts", explaintext=1, exsectionformat="wiki",
            titles=titolo, redirects=1)
    pg = d["query"]["pages"][0]
    if pg.get("missing"):
        return ""
    tx = pg.get("extract", "") or ""
    fn.write_bytes(tx.encode("utf-8"))
    return tx

GREZZI = {}
print(f"scarico {len(ARGOMENTI)} articoli (uno per richiesta, ~0.3 s ciascuno)...")
for i, t in enumerate(ARGOMENTI, 1):
    try:
        tx = scarica(t)
        if tx:
            GREZZI[t] = tx
    except Exception as e:
        print(f"\n  errore su '{t}': {type(e).__name__}: {e}")
    if i % 20 == 0:
        print(f"  {i}/{len(ARGOMENTI)}", end="", flush=True)
print(f"\narticoli scaricati: {len(GREZZI)}/{len(ARGOMENTI)}")

## 4. Pulizia e gate di qualità

L'estratto contiene, oltre al corpo, l'apparato finale (*See also*, *References*, *External
links*…) che è fatto di elenchi, non di prosa. Si taglia alla prima intestazione di apparato e
si eliminano le righe di intestazione, poi si applicano gli stessi gate del notebook
principale. Gli articoli che restano vengono **ordinati per lunghezza di prosa reale** e si
tengono i primi `N_ARTICOLI`.

In [ ]:
APPARATO = re.compile(
    r"^==+\s*(See also|References|Notes|Citations|Sources|Bibliography|Further reading|"
    r"External links|Works cited|Footnotes|Explanatory notes|General sources)\s*==+\s*$",
    re.M | re.I)
INTESTAZIONE = re.compile(r"^==+.*?==+\s*$", re.M)
FRASE = re.compile(r"[.!?][\"'\u2019\)\]]*(?:\s|$)")

def pulisci(tx):
    m = APPARATO.search(tx)
    if m:
        tx = tx[:m.start()]
    tx = INTESTAZIONE.sub("", tx)                 # via le intestazioni di sezione
    tx = tx.replace("\r\n", "\n").replace("\r", "\n")
    tx = re.sub(r"\n[ \t]+\n", "\n\n", tx)
    return re.sub(r"\n{3,}", "\n\n", tx).strip()

def qualita(t):
    toks = [w.lower() for w in TOKEN_RE.findall(t)]
    grams = [t[i:i+10] for i in range(0, max(len(t)-10, 0), 3)]
    righe = [L for L in t.split("\n") if L.strip()]
    return dict(
        n_char=len(t),
        frasi_1k=1000*len(FRASE.findall(t))/len(t) if t else np.nan,
        righe_corte=sum(1 for L in righe if len(L) < 80)/len(righe) if righe else np.nan,
        len_token=float(np.mean([len(w) for w in toks])) if toks else np.nan,
        kgram=len(set(grams))/len(grams) if grams else np.nan)

righe = []
TESTI = {}
for t, grezzo in GREZZI.items():
    p = pulisci(grezzo)
    TESTI[t] = p
    q = qualita(p); q["titolo"] = t; q["n_grezzo"] = len(grezzo)
    righe.append(q)
Q = pd.DataFrame(righe).set_index("titolo")
Q["abbastanza_lungo"] = Q["n_char"] >= N_EFF
Q["prosa"] = ((Q["frasi_1k"] >= FRASI_1K_MIN) & (Q["righe_corte"] <= RIGHE_CORTE_MAX)
              & (Q["len_token"] <= LEN_TOKEN_MAX))
Q["ammesso"] = Q["abbastanza_lungo"] & Q["prosa"]

print(f"articoli: {len(Q)} | con >= {N_EFF:,} caratteri di prosa: {int(Q['abbastanza_lungo'].sum())}"
      f" | che passano i gate: {int(Q['ammesso'].sum())}")
_sc = Q[Q["abbastanza_lungo"] & ~Q["prosa"]]
if len(_sc):
    print(f"\nscartati dai gate pur essendo lunghi ({len(_sc)}):")
    print(_sc[["n_char", "frasi_1k", "righe_corte", "len_token"]].round(2).to_string())

SCELTI = Q[Q["ammesso"]].sort_values("n_char", ascending=False).head(N_ARTICOLI)
print(f"\nselezionati {len(SCELTI)} articoli, da {SCELTI['n_char'].min():,} a "
      f"{SCELTI['n_char'].max():,} caratteri di prosa")
print(SCELTI[["n_char", "frasi_1k", "righe_corte", "len_token", "kgram"]]
      .head(15).round(2).to_string())
Q.to_csv(OUT_DIR / "dati" / "articoli_qualita.csv")

## 5. Riferimento letterario alla stessa lunghezza

Gli stessi 20 segmenti di *Guerra e pace* del notebook principale, da `N_EFF` caratteri. Il
confronto ha senso solo a lunghezza uguale: $\sigma_\tau/\langle\tau\rangle$ diverge con $N$
per code con $\mu < 3$.

In [ ]:
GUT_START = re.compile(r"\*\*\*\s*START OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*", re.S)
GUT_END   = re.compile(r"\*\*\*\s*END OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*", re.S)
STRUCT = re.compile(
    r"^[ \t]*(?:BOOK\s+[A-Z]+[^\n]*|CHAPTER\s+[IVXLCDM\d]+[^\n]*|"
    r"(?:FIRST|SECOND)\s+EPILOGUE[^\n]*|EPILOGUE[^\n]*|CONTENTS[^\n]*|"
    r"PART\s+[IVXLCDM\d]+[^\n]*|APPENDIX[^\n]*|\d+)[ \t]*$", re.M)

def fetch_libro(url):
    fn = CACHE_LIB / (hashlib.sha256(url.encode()).hexdigest()[:12] + ".txt")
    if fn.exists():
        raw = fn.read_bytes().decode("utf-8", errors="replace")
        if "\r\r" not in raw:
            print(f"corpus letterario dalla cache: {fn}")
            return raw
        print("cache corrotta (newline raddoppiati): riscarico")
    req = urllib.request.Request(url, headers={"User-Agent": UA})
    with urllib.request.urlopen(req, timeout=90, context=CTX) as r:
        raw = r.read().decode("utf-8", errors="replace")
    CACHE_LIB.mkdir(parents=True, exist_ok=True)
    fn.write_bytes(raw.encode("utf-8"))
    return raw

def clean_literary(raw, start_phrase):
    raw = raw.replace("\r\n", "\n").replace("\r", "\n")
    m = GUT_START.search(raw)
    if m: raw = raw[m.end():]
    m = GUT_END.search(raw)
    if m: raw = raw[:m.start()]
    i = raw.find(start_phrase)
    if i < 0: raise RuntimeError("START_PHRASE non trovata")
    raw = STRUCT.sub("", raw[i:])
    raw = re.sub(r"\n[ \t]+\n", "\n\n", raw)
    return re.sub(r"\n{3,}", "\n\n", raw).strip()

CORPUS = clean_literary(fetch_libro(GUTENBERG_URL), START_PHRASE)
BODY = CORPUS[PROMPT_CHARS:]
starts = np.linspace(0, max(len(BODY) - N_EFF, 0), N_SEG_LETTERARI).astype(int)
SEGMENTI = [BODY[s:s + N_EFF] for s in starts]
print(f"corpus letterario: {len(BODY):,} caratteri -> {len(SEGMENTI)} segmenti da {N_EFF:,}")

# le due sorgenti hanno la stessa densita' di spazi bianchi?
def dens(t):
    return dict(newline=t.count("\n")/len(t), capoverso=t.count("\n\n")/len(t),
                spazio=t.count(" ")/len(t))
_dw = pd.DataFrame([dens(TESTI[t][:N_EFF]) for t in SCELTI.index]).mean()
_dl = pd.DataFrame([dens(s) for s in SEGMENTI]).mean()
CMP = pd.DataFrame({"letterario": _dl, "wikipedia": _dw})
CMP["rapporto"] = CMP["wikipedia"] / CMP["letterario"]
print("\ndensita' di spazi bianchi")
print(CMP.round(4).to_string())
# Gutenberg va a capo ogni ~72 caratteri, Wikipedia solo a fine capoverso: il rapporto
# degli a-capo e' ~0.12, quello degli spazi ~0.97. La prima versione di questo controllo
# guardava solo gli spazi e passava senza accorgersene, quindi ora si guardano entrambi.
#
# Perche' NON invalida il risultato: sigma_tau/<tau> e' un RAPPORTO, quindi invariante
# per dilatazione uniforme dell'asse dei tempi. Togliere gli a-capo comprime tutti i tau
# dello stesso fattore e il coefficiente di variazione non cambia. Verificato srotolando
# i capoversi di entrambe le sorgenti: 103/140 = 0.736 e 190/280 = 0.679 prima e dopo,
# con cv_tau medio delle keyword identico alla terza cifra (1.619 e 1.505).
# Vale per cv_tau, cioe' per il verdetto. Per gamma il range di fit e' fissato in
# CARATTERI, quindi srotolare sposta un po' le scale fisiche sondate: se si vuole
# confrontare gamma fra sorgenti con formattazione diversa, conviene srotolare entrambe.
def srotola(t):
    """unisce le righe dentro un capoverso (toglie l'a-capo forzato)"""
    return re.sub(r"(?<!\n)\n(?!\n)", " ", t)

for _k in ["spazio", "newline"]:
    _r = CMP.loc[_k, "rapporto"]
    if not (0.6 < _r < 1.6):
        print(f"NOTA: densita' di '{_k}' molto diversa fra le due sorgenti (rapporto {_r:.2f}).")
        if _k == "newline":
            print("      Atteso: e' la formattazione di Gutenberg. Non tocca cv_tau (invariante")
            print("      di scala); per confrontare gamma usare srotola() su entrambe.")
        else:
            print("      QUESTO conta: gli spazi sono una delle 41 sequenze analizzate.")

## 6. Analisi

In [ ]:
REC = []
print("Wikipedia: ", end="", flush=True)
for t in SCELTI.index:
    REC += analizza_testo(TESTI[t], dict(label=t, gruppo="wikipedia", fonte=t))
    print(".", end="", flush=True)
print(" fatto")
print("letterario: ", end="", flush=True)
for i, s in enumerate(SEGMENTI):
    REC += analizza_testo(s, dict(label=f"wrnpc_{i}", gruppo="letterario", fonte="Guerra e pace"))
    print(".", end="", flush=True)
print(" fatto")

A = pd.DataFrame(REC)
A.to_csv(OUT_DIR / "dati" / "risultati_wikipedia.csv", index=False)
print(f"\nrecord: {len(A):,}")
print(A.groupby(["gruppo", "tipo"])["stimabile"].agg(["size", "mean"]).round(3).to_string())
print("\ncontrollo del null model A1 (atteso ~1.00):")
print(A[A["stimabile"]].groupby(["gruppo", "livello"])["gamma_A1"].mean().round(3).to_string())

## 7. Il test: le keyword sono più bursty dei controlli appaiati?

Appaiamento **posizionale** dentro ogni testo: `select_targets` costruisce `matched[i]` come il
controllo a frequenza più vicina a `keys[i]`, e i record sono accodati in quell'ordine. Non si
filtra su `stimabile` prima di appaiare, altrimenti le coppie si sfasano.

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k/n; d = 1 + z**2/n
    c = (p + z**2/(2*n))/d
    h = z*np.sqrt(p*(1-p)/n + z**2/(4*n**2))/d
    return (max(c-h, 0.0), min(c+h, 1.0))

def test_appaiato(sub, col):
    vinte = tot = 0; diffs = []
    for _, doc in sub.groupby("label"):
        k = doc[doc["tipo"] == "keyword"].sort_index()
        m = doc[doc["tipo"] == "appaiata"].sort_index()
        for ik, im in zip(k.index, m.index):
            va, vb = A.at[ik, col], A.at[im, col]
            if np.isfinite(va) and np.isfinite(vb):
                tot += 1; vinte += int(va > vb); diffs.append(va - vb)
    lo, hi = wilson(vinte, tot)
    p = float(sps.binomtest(vinte, tot, 0.5).pvalue) if (HAS_SCIPY and tot) else np.nan
    return dict(vittorie=vinte, confronti=tot, frazione=vinte/tot if tot else np.nan,
                ic_lo=lo, ic_hi=hi, p=p,
                diff_media=float(np.mean(diffs)) if diffs else np.nan)

righe = []
for g, sub in A.groupby("gruppo"):
    for col in ["cv_tau", "gamma"]:
        r = test_appaiato(sub, col); r.update(gruppo=g, metrica=col,
                                              documenti=sub["label"].nunique())
        righe.append(r)
PT = pd.DataFrame(righe)[["gruppo", "metrica", "documenti", "vittorie", "confronti",
                          "frazione", "ic_lo", "ic_hi", "p", "diff_media"]]
PT.to_csv(OUT_DIR / "dati" / "test_appaiato.csv", index=False)
print("Le keyword battono il proprio controllo a frequenza appaiata?\n")
for _, x in PT.iterrows():
    st = "" if not np.isfinite(x["p"]) else (" ***" if x["p"] < 1e-3 else
         " **" if x["p"] < 1e-2 else " *" if x["p"] < .05 else "  n.s.")
    print(f"  {x['gruppo']:12s} {x['metrica']:7s} {int(x['vittorie']):4d}/{int(x['confronti']):4d}"
          f" = {x['frazione']:.2f} [{x['ic_lo']:.2f}, {x['ic_hi']:.2f}]"
          f"  su {int(x['documenti'])} testi  p={x['p']:.1e}{st}")

In [ ]:
# --- livelli linguistici e ricorrenza ---
LIV = [("lettera", "lettere"), ("funzione", "parole funzione"),
       ("appaiata", "controlli appaiati"), ("keyword", "keyword")]

def per_doc(sub, tipo, col):
    return sub[sub["tipo"] == tipo].groupby("label")[col].mean().dropna()

righe = []
for tipo, nome in LIV:
    for g, sub in A.groupby("gruppo"):
        v = per_doc(sub, tipo, "cv_tau"); w = per_doc(sub, tipo, "gamma")
        righe.append(dict(tipo=tipo, gruppo=g, n_doc=len(v),
                          cv=v.mean(), cv_sd=v.std(), gamma=w.mean(), gamma_sd=w.std()))
LEVELS = pd.DataFrame(righe)
for tipo, nome in LIV:
    s = LEVELS[LEVELS["tipo"] == tipo].set_index("gruppo")
    if {"wikipedia", "letterario"} <= set(s.index) and HAS_SCIPY:
        a = per_doc(A[A.gruppo == "wikipedia"], tipo, "cv_tau")
        b = per_doc(A[A.gruppo == "letterario"], tipo, "cv_tau")
        if len(a) >= 3 and len(b) >= 3:
            LEVELS.loc[(LEVELS["tipo"] == tipo) & (LEVELS["gruppo"] == "wikipedia"), "p_vs_lett"] = \
                float(sps.mannwhitneyu(a, b, alternative="two-sided").pvalue)
LEVELS.to_csv(OUT_DIR / "dati" / "livelli.csv", index=False)

print("Burstiness sigma_tau/<tau> per livello (media fra testi, N =", f"{N_EFF:,} caratteri)\n")
for tipo, nome in LIV:
    print(f"  {nome}")
    for _, x in LEVELS[LEVELS["tipo"] == tipo].iterrows():
        pp = f"  p={x['p_vs_lett']:.3f}" if ("p_vs_lett" in x and np.isfinite(x.get("p_vs_lett", np.nan))) else ""
        print(f"    {x['gruppo']:12s} {x['cv']:.2f} +- {x['cv_sd']:.2f}  ({int(x['n_doc'])} testi){pp}")

print("\nRicorrenza della keyword piu' frequente (una parola bursty deve prima ricorrere):")
for g, sub in A.groupby("gruppo"):
    top = sub[sub["tipo"] == "keyword"].groupby("label")["n_eventi"].max()
    ev = sub[sub["tipo"] == "keyword"]["n_eventi"]
    print(f"  {g:12s} mediana {top.median():5.0f}  [{top.min():.0f}-{top.max():.0f}]"
          f"  | occorrenze mediane {ev.median():4.0f}"
          f"  | sopra MIN_EVENTS {100*(ev >= MIN_EVENTS).mean():3.0f}%")

## 8. Figure

In [ ]:
COL = {"letterario": "#333333", "wikipedia": "#0072B2"}
fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.8))

ax = axes[0]
ys = list(PT[PT["metrica"] == "cv_tau"].itertuples())
for i, x in enumerate(ys):
    err = np.array([[max(x.frazione - x.ic_lo, 0)], [max(x.ic_hi - x.frazione, 0)]])
    ax.errorbar([x.frazione], [i], xerr=err, fmt="o", ms=10, capsize=5,
                color=COL.get(x.gruppo, "grey"), lw=2)
ax.set_yticks(range(len(ys))); ax.set_yticklabels([x.gruppo for x in ys])
ax.set_ylim(-0.6, len(ys) - 0.4); ax.set_xlim(0, 1)
ax.axvline(0.5, color="grey", ls=":", lw=1.4)
ax.set_xlabel("frazione di coppie con keyword più bursty")
ax.set_title("A) l'effetto di Altmann esiste?\n(0.5 = nessun effetto)", fontsize=10)

ax = axes[1]
tipi = [t for t, _ in LIV]; xs = np.arange(len(tipi))
for g in ["letterario", "wikipedia"]:
    s = LEVELS[LEVELS["gruppo"] == g].set_index("tipo").reindex(tipi)
    ax.errorbar(xs + (.05 if g == "wikipedia" else -.05), s["cv"], yerr=s["cv_sd"],
                marker="s" if g == "letterario" else "o", ms=8, lw=2, capsize=4,
                color=COL[g], label=g)
ax.axhline(1, color="grey", ls=":", lw=1.2)
ax.set_xticks(xs); ax.set_xticklabels([n for _, n in LIV], rotation=12, fontsize=8)
ax.set_ylabel(r"$\sigma_\tau/\langle\tau\rangle$")
ax.set_title("B) la scala della burstiness per livello", fontsize=10)
ax.legend(fontsize=8)

ax = axes[2]
dati, et = [], []
for g in ["letterario", "wikipedia"]:
    v = per_doc(A[A.gruppo == g], "keyword", "cv_tau")
    if len(v): dati.append(v.values); et.append(f"{g}\n({len(v)})")
if dati:
    bp = ax.boxplot(dati, tick_labels=et, showmeans=True, widths=.5, patch_artist=True)
    for b, g in zip(bp["boxes"], ["letterario", "wikipedia"]):
        b.set(facecolor=COL[g], alpha=.30)
    for i, v in enumerate(dati):
        ax.scatter(np.full(len(v), i+1) + rng.normal(0, .05, len(v)), v,
                   s=16, color="k", alpha=.5, zorder=4)
ax.axhline(1, color="grey", ls=":", lw=1.2)
ax.set_ylabel(r"$\sigma_\tau/\langle\tau\rangle$ delle keyword")
ax.set_title("C) burstiness delle keyword per testo", fontsize=10)

fig.suptitle(f"La prosa enciclopedica regge il protocollo di Altmann?  "
             f"({len(SCELTI)} articoli vs {len(SEGMENTI)} segmenti letterari, "
             f"N = {N_EFF:,} caratteri)", fontweight="bold")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure" / "wiki_altmann.png"); plt.show()

## 9. Verdetto

In [ ]:
L = []
L.append("PROSA ENCICLOPEDICA E PROTOCOLLO DI ALTMANN — test di fattibilità")
L.append("=" * 70)
L.append(f"articoli Wikipedia analizzati: {len(SCELTI)} | segmenti letterari: {len(SEGMENTI)}")
L.append(f"lunghezza comune: {N_EFF:,} caratteri | fit di gamma su t in {FIT_RANGE}")
L.append("")
for _, x in PT.iterrows():
    L.append(f"  {x['gruppo']:12s} {x['metrica']:7s}: {int(x['vittorie']):4d}/{int(x['confronti']):4d}"
             f" = {x['frazione']:.2f} [{x['ic_lo']:.2f}, {x['ic_hi']:.2f}]  p={x['p']:.1e}")
L.append("")
w = PT[(PT["gruppo"] == "wikipedia") & (PT["metrica"] == "cv_tau")]
l = PT[(PT["gruppo"] == "letterario") & (PT["metrica"] == "cv_tau")]
if len(w) and len(l):
    fw, lo_w = float(w["frazione"].iloc[0]), float(w["ic_lo"].iloc[0])
    fl = float(l["frazione"].iloc[0])
    L.append("VERDETTO sulla burstiness delle keyword:")
    if lo_w > 0.5:
        L.append(f"  L'effetto ESISTE in Wikipedia ({fw:.2f}, IC inferiore {lo_w:.2f} > 0.5).")
        L.append(f"  Nel letterario vale {fl:.2f}. Il confronto con Grokipedia ha un segnale")
        L.append("  da rilevare: si puo' procedere.")
        if fw < fl - 0.10:
            L.append(f"  Attenzione: l'effetto e' piu' debole del letterario ({fw:.2f} vs {fl:.2f}),")
            L.append("  quindi servira' piu' potenza, cioe' piu' coppie di articoli.")
    else:
        L.append(f"  L'effetto NON e' dimostrato in Wikipedia ({fw:.2f}, IC [{lo_w:.2f}, "
                 f"{float(w['ic_hi'].iloc[0]):.2f}] include 0.5).")
        L.append(f"  Nel letterario alla stessa lunghezza vale {fl:.2f}.")
        L.append("  Senza effetto nel riferimento umano non c'e' nulla che il confronto con")
        L.append("  Grokipedia possa rilevare: la strada enciclopedica non regge, oppure")
        L.append("  servono articoli piu' lunghi (N_EFF piu' alto) o piu' numerosi.")
k = LEVELS[LEVELS["tipo"] == "keyword"].set_index("gruppo")
if {"wikipedia", "letterario"} <= set(k.index):
    L.append("")
    L.append(f"  burstiness delle keyword: wikipedia {k.loc['wikipedia','cv']:.2f} "
             f"vs letterario {k.loc['letterario','cv']:.2f} "
             f"(rapporto {k.loc['wikipedia','cv']/k.loc['letterario','cv']:.2f})")
sintesi = "\n".join(L)
(OUT_DIR / "verdetto.txt").write_text(sintesi, encoding="utf-8")
print(sintesi)